In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Check GPU**

In [1]:
import torch
print("GPU:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Torch:", torch.__version__)

GPU: True
Device: Tesla T4
Torch: 2.9.0+cu126


**Clone YOLOv7 and install**

In [2]:
import os

os.system("git clone https://github.com/WongKinYiu/yolov7")
os.system("pip install -q -r /kaggle/working/yolov7/requirements.txt")
os.system("pip install -q wandb==0.13.0")  # pin old wandb to avoid login issues
print("Done!")

Cloning into 'yolov7'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 74.5 MB/s eta 0:00:00


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


Done!


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


**Patch torch.load in ALL yolov7 files**

In [3]:
import os
import glob

yolov7_dir = "/kaggle/working/yolov7"

# Find all python files in yolov7
py_files = glob.glob(f"{yolov7_dir}/**/*.py", recursive=True)

patched = 0
for filepath in py_files:
    with open(filepath, "r") as f:
        content = f.read()

    if "torch.load(" not in content:
        continue

    new_content = content

    # Fix pattern 1: torch.load(x, map_location=y)
    import re
    new_content = re.sub(
        r'torch\.load\((.+?),\s*map_location=(.+?)\)(?!\s*\[)',
        r'torch.load(\1, map_location=\2, weights_only=False)',
        new_content
    )
    # Fix pattern 2: torch.load(x, map_location=y)['something']
    new_content = re.sub(
        r'torch\.load\((.+?),\s*map_location=(.+?)\)(\[)',
        r'torch.load(\1, map_location=\2, weights_only=False)\3',
        new_content
    )
    # Fix pattern 3: plain torch.load(x) with no map_location
    new_content = re.sub(
        r'torch\.load\(([^,)]+)\)(?!.*weights_only)',
        r'torch.load(\1, weights_only=False)',
        new_content
    )

    # Remove any accidental duplicates
    new_content = new_content.replace(
        'weights_only=False, weights_only=False',
        'weights_only=False'
    )

    if new_content != content:
        with open(filepath, "w") as f:
            f.write(new_content)
        patched += 1
        print(f"Patched: {filepath.replace(yolov7_dir, '')}")

print(f"\nTotal files patched: {patched}")

# Verify train.py
print("\ntrain.py torch.load lines:")
with open(f"{yolov7_dir}/train.py") as f:
    for i, line in enumerate(f, 1):
        if "torch.load" in line:
            print(f"  Line {i}: {line.strip()}")

Patched: /train_aux.py
Patched: /detect.py
Patched: /train.py
Patched: /hubconf.py
Patched: /utils/datasets.py
Patched: /utils/general.py
Patched: /utils/aws/resume.py
Patched: /models/experimental.py

Total files patched: 8

train.py torch.load lines:
  Line 71: run_id = torch.load(weights, map_location=device, weights_only=False).get('wandb_id') if weights.endswith('.pt') and os.path.isfile(weights) else None
  Line 87: ckpt = torch.load(weights, map_location=device, weights_only=False)  # load checkpoint


**Explore your dataset structure**

In [4]:
import os

DATASET_PATH = "/kaggle/input/datasets/lokiusevoldemort/candy-data"

print("Dataset contents:")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:5]:
            print(f"  {indent}{f}")
        if len(files) > 5:
            print(f"  {indent}... and {len(files)-5} more")

Dataset contents:
candy-data/
  notes.json
  classes.txt
  labels/
    356f8bbd-candy_53.txt
    84f6fae9-candy_86.txt
    83264950-candy_100.txt
    213e74f1-candy_35.txt
    f1bb58c1-candy_1.txt
    ... and 157 more
  images/
    426f5963-candy_45.jpg
    e365fad1-candy_111.jpg
    e4becfac-candy_153.jpg
    63b1fca9-candy_2.jpg
    e4b09525-candy_7.jpg
    ... and 157 more


**Set up paths based on your structure**

In [5]:
import os

DATASET_PATH = "/kaggle/input/datasets/lokiusevoldemort/candy-data"

# These are your actual paths
IMAGES_PATH  = os.path.join(DATASET_PATH, "images")
LABELS_PATH  = os.path.join(DATASET_PATH, "labels")
CLASSES_PATH = os.path.join(DATASET_PATH, "classes.txt")

# Verify
print("Images path exists :", os.path.exists(IMAGES_PATH))
print("Labels path exists :", os.path.exists(LABELS_PATH))
print("Classes file exists:", os.path.exists(CLASSES_PATH))
print("Images count       :", len(os.listdir(IMAGES_PATH)) if os.path.exists(IMAGES_PATH) else 0)
print("Labels count       :", len(os.listdir(LABELS_PATH)) if os.path.exists(LABELS_PATH) else 0)

# Read and print classes
with open(CLASSES_PATH) as f:
    classes = [l.strip() for l in f if l.strip()]
print(f"\nClasses ({len(classes)}):")
for i, c in enumerate(classes):
    print(f"  {i}: {c}")

Images path exists : True
Labels path exists : True
Classes file exists: True
Images count       : 162
Labels count       : 162

Classes (11):
  0: MMs_peanut
  1: MMs_regular
  2: airheads
  3: gummy_worms
  4: milky_way
  5: nerds
  6: skittles
  7: snickers
  8: starbust
  9: three_musketeers
  10: twizzlers


**GAN Training**

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import os
from tqdm import tqdm

class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x):
        return self.main(x)

class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)

class CandyDataset(Dataset):
    def __init__(self, images_dir, image_size=64):
        self.images_dir = images_dir
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
        ])
        self.files = [f for f in os.listdir(images_dir)
                      if f.lower().endswith(('.jpg','.jpeg','.png'))]
        print(f"Found {len(self.files)} images for GAN training")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.images_dir, self.files[idx])).convert('RGB')
        return self.transform(img)

def train_gan(images_dir, output_dir, n_epochs=100, n_generate=800):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training GAN on {device}")
    os.makedirs(output_dir, exist_ok=True)

    dataset    = CandyDataset(images_dir)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

    G = Generator().to(device)
    D = Discriminator().to(device)

    criterion = nn.BCELoss()
    optG = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
    optD = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

    for epoch in range(n_epochs):
        for real_imgs in dataloader:
            real_imgs = real_imgs.to(device)
            b = real_imgs.size(0)

            # Train D
            D.zero_grad()
            loss_D_real = criterion(D(real_imgs), torch.ones(b, device=device))
            noise = torch.randn(b, 100, 1, 1, device=device)
            fake  = G(noise)
            loss_D_fake = criterion(D(fake.detach()), torch.zeros(b, device=device))
            (loss_D_real + loss_D_fake).backward()
            optD.step()

            # Train G
            G.zero_grad()
            loss_G = criterion(D(fake), torch.ones(b, device=device))
            loss_G.backward()
            optG.step()

        if (epoch+1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{n_epochs}]  "
                  f"Loss_D: {(loss_D_real+loss_D_fake).item():.4f}  "
                  f"Loss_G: {loss_G.item():.4f}")

    # Generate images
    print(f"\nGenerating {n_generate} images...")
    G.eval()
    count = 0
    with torch.no_grad():
        while count < n_generate:
            batch = min(64, n_generate - count)
            noise = torch.randn(batch, 100, 1, 1, device=device)
            fake  = G(noise)
            fake  = (fake * 0.5 + 0.5).clamp(0, 1)
            for j in range(batch):
                img = fake[j].cpu().permute(1,2,0).numpy()
                img = Image.fromarray((img*255).astype(np.uint8)).resize((416,416))
                img.save(os.path.join(output_dir, f"gan_{count:04d}.jpg"))
                count += 1

    print(f"Saved {count} GAN images to {output_dir}")

GAN_OUTPUT = "/kaggle/working/gan_generated"
train_gan(IMAGES_PATH, GAN_OUTPUT, n_epochs=100, n_generate=800)
print("GAN done!")

Training GAN on cuda
Found 162 images for GAN training
Epoch [10/100]  Loss_D: 0.6035  Loss_G: 10.8603
Epoch [20/100]  Loss_D: 0.3505  Loss_G: 8.2029
Epoch [30/100]  Loss_D: 0.1242  Loss_G: 5.2046
Epoch [40/100]  Loss_D: 2.8194  Loss_G: 1.9525
Epoch [50/100]  Loss_D: 1.4218  Loss_G: 1.8578
Epoch [60/100]  Loss_D: 0.3288  Loss_G: 4.0590
Epoch [70/100]  Loss_D: 1.9098  Loss_G: 1.3750
Epoch [80/100]  Loss_D: 0.1233  Loss_G: 6.3817
Epoch [90/100]  Loss_D: 0.4317  Loss_G: 4.4319
Epoch [100/100]  Loss_D: 0.5429  Loss_G: 8.4054

Generating 800 images...
Saved 800 GAN images to /kaggle/working/gan_generated
GAN done!


**Build YOLO dataset with train/val split**

In [7]:
import os
import shutil
import random
from pathlib import Path

YOLO_DIR = "/kaggle/working/candy_yolo"

# Create folders
for split in ["train", "val"]:
    os.makedirs(f"{YOLO_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_DIR}/labels/{split}",  exist_ok=True)

# Get all images
all_images = sorted([
    f for f in os.listdir(IMAGES_PATH)
    if f.lower().endswith(('.jpg','.jpeg','.png'))
])

# 80/20 split
random.seed(42)
random.shuffle(all_images)
split_idx  = int(len(all_images) * 0.8)
train_imgs = all_images[:split_idx]
val_imgs   = all_images[split_idx:]

# Copy real images + labels
def copy_pair(img_name, split):
    stem = Path(img_name).stem
    shutil.copy2(
        os.path.join(IMAGES_PATH, img_name),
        f"{YOLO_DIR}/images/{split}/{img_name}"
    )
    lbl = os.path.join(LABELS_PATH, stem + ".txt")
    dst = f"{YOLO_DIR}/labels/{split}/{stem}.txt"
    if os.path.exists(lbl):
        shutil.copy2(lbl, dst)
    else:
        open(dst, 'w').close()

for img in train_imgs: copy_pair(img, "train")
for img in val_imgs:   copy_pair(img, "val")

# Add GAN images to train (background images, no labels)
gan_imgs = sorted(os.listdir(GAN_OUTPUT))[:400]
for img_name in gan_imgs:
    shutil.copy2(
        os.path.join(GAN_OUTPUT, img_name),
        f"{YOLO_DIR}/images/train/{img_name}"
    )
    open(f"{YOLO_DIR}/labels/train/{Path(img_name).stem}.txt", 'w').close()

print(f"Train images : {len(os.listdir(f'{YOLO_DIR}/images/train'))}")
print(f"Val images   : {len(os.listdir(f'{YOLO_DIR}/images/val'))}")

Train images : 529
Val images   : 33


**Write candy.yaml with absolute paths**

In [8]:
import yaml

config = {
    "train": "/kaggle/working/candy_yolo/images/train",
    "val":   "/kaggle/working/candy_yolo/images/val",
    "nc":    len(classes),
    "names": classes,
}

with open("/kaggle/working/candy.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("candy.yaml written:")
with open("/kaggle/working/candy.yaml") as f:
    print(f.read())

candy.yaml written:
names:
- MMs_peanut
- MMs_regular
- airheads
- gummy_worms
- milky_way
- nerds
- skittles
- snickers
- starbust
- three_musketeers
- twizzlers
nc: 11
train: /kaggle/working/candy_yolo/images/train
val: /kaggle/working/candy_yolo/images/val



**Download pretrained weights**

In [9]:
import os
os.chdir("/kaggle/working/yolov7")
os.system("wget -q https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt")
print("Weights downloaded:", os.path.exists("yolov7.pt"))
print("Size:", os.path.getsize("yolov7.pt") // (1024*1024), "MB")

Weights downloaded: True
Size: 72 MB


**TRAIN**

In [10]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"]     = "disabled"
os.chdir("/kaggle/working/yolov7")

os.system("""WANDB_DISABLED=true python train.py \
    --weights yolov7.pt \
    --cfg cfg/training/yolov7.yaml \
    --data /kaggle/working/candy.yaml \
    --epochs 100 \
    --batch-size 16 \
    --img 640 \
    --workers 4 \
    --name candy_yolov7 \
    --hyp data/hyp.scratch.p5.yaml \
    --project /kaggle/working/runs/train \
    --exist-ok \
    --cache""")

2026-03-17 19:24:42.457171: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773775482.635688     973 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773775482.685767     973 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773775483.081741     973 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773775483.081782     973 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773775483.081786     973 computation_placer.cc:177] computation placer alr


autoanchor: Analyzing anchors... anchors/target = 5.77, Best Possible Recall (BPR) = 0.9980


      0/99    0.998G   0.07863   0.01365   0.03792    0.1302         0       640: 100%|██████████| 34/34 [02:04<00:00,  3.66s/it]
               Class      Images      Labels           P           R      mAP@.5  mAP@.5:.95:   0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
               Class      Images      Labels           P           R      mAP@.5  mAP@.5:.95: 100%|██████████| 2/2 [00:05<00:00,  2.58s/it]

     Epoch   gpu_mem       box       obj       cls     total    labels  img_size
  0%|          | 0/34 [00:00<?, ?it/s]

                 all          33         125      0.0019      0.0157    0.000565    0.000101


      1/99     6.34G   0.06653  0.009993   0.03665    0.1132         0       640: 100%|██████████| 34/34 [01:09<00:00,  2.06s/it]
               Class      Images      Labels           P           R      mAP@.5  mAP@.5:.95: 100%|██████████| 2/2 [00:00<00:00,  3.84it/s]

     Epoch   gpu_mem       box       obj       cls     total    labels  img_size
  0%|          | 0/34 [00:00<?, ?it/s]

...

     98/99     6.83G   0.01362  0.002788  0.003338   0.01975         0       640: 100%|██████████| 34/34 [01:11<00:00,  2.11s/it]
               Class      Images      Labels           P           R      mAP@.5  mAP@.5:.95: 100%|██████████| 2/2 [00:00<00:00,  3.83it/s]

     Epoch   gpu_mem       box       obj       cls     total    labels  img_size
  0%|          | 0/34 [00:00<?, ?it/s]

                 all          33         125        0.97        0.94       0.972        0.87


     99/99      7.2G   0.01387  0.002753  0.003027   0.01965         2       640: 100%|██████████| 34/34 [01:12<00:00,  2.12s/it]
               Class      Images      Labels           P           R      mAP@.5  mAP@.5:.95: 100%|██████████| 2/2 [00:03<00:00,  1.69s/it]
100 epochs completed in 2.067 hours.

Traceback (most recent call last):
  File "/kaggle/working/yolov7/train.py", line 616, in <module>
    train(hyp, opt, device, tb_writer)
  File "/kaggle/working/yolov7/train.py", line 513, in train
    strip_optimizer(f)  # strip optimizers
    ^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/yolov7/utils/general.py", line 802, in strip_optimizer
    x = torch.load(f, map_location=torch.device('cpu', weights_only=False))
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: device() got an unexpected keyword argument 'weights_only'


                 all          33         125        0.97       0.937       0.973       0.877
          MMs_peanut          33           9       0.997           1       0.995       0.868
         MMs_regular          33          11       0.987           1       0.995       0.912
            airheads          33          21       0.884       0.952       0.959       0.881
         gummy_worms          33          13       0.994           1       0.995       0.925
           milky_way          33           9           1       0.845         0.9       0.775
               nerds          33          10           1       0.969       0.995       0.896
            skittles          33          11       0.915           1       0.995       0.917
            snickers          33          11           1       0.837       0.971       0.869
            starbust          33          13       0.923       0.923        0.93       0.829
    three_musketeers          33           9           1       0.778  

256

**Save outputs for download**

In [11]:
import shutil, os

OUT = "/kaggle/working/final_outputs"
os.makedirs(OUT, exist_ok=True)

shutil.copy2("/kaggle/working/runs/train/candy_yolov7/weights/best.pt", f"{OUT}/candy_best.pt")
shutil.copy2("/kaggle/working/runs/train/candy_yolov7/weights/last.pt", f"{OUT}/candy_last.pt")
shutil.copy2("/kaggle/working/candy.yaml", f"{OUT}/candy.yaml")
shutil.copy2(CLASSES_PATH, f"{OUT}/classes.txt")

print("Files ready to download:")
for f in os.listdir(OUT):
    size = os.path.getsize(os.path.join(OUT, f))
    print(f"  {f:30s} {size//1024:>6} KB")

Files ready to download:
  classes.txt                         0 KB
  candy_last.pt                  291821 KB
  candy.yaml                          0 KB
  candy_best.pt                  291820 KB
